# DWBI Week 4 — Dimensional Modeling
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's Week-3 star (1 fact + 5 dims, 34 rows, 578,650 kg) faces the snowflake question: should DimCity be normalized? Seven tasks count the joins, build the snowflake, do the storage arithmetic, register the relationships, settle cross-filter direction — and the totals do not move by a single kilogram.

## T4 — Relationship register (20 marks)

Specified **before** Power BI opens — the register comes first, the clicking comes second.

![The Week-4 star](charts/t4_model.png)

| # | One side (dimension) | Many side (fact) | Cardinality | Cross-filter | Active? | Why |
|---|---|---|---|---|---|---|
| 1 | DimDate[DateKey] | FactShipment[DateKey] | * : 1 | Single | Yes | Dates the shipment leg |
| 2 | DimCity[CityKey] | FactShipment[OriginCityKey] | * : 1 | Single | Yes | Origin role: where each leg started |
| 3 | DimCity[CityKey] | FactShipment[DestCityKey] | * : 1 | Single | **No** | Destination role — inactive: one dimension cannot have two active paths to the same fact |
| 4 | DimRoute[RouteKey] | FactShipment[RouteKey] | * : 1 | Single | Yes | The corridor each leg ran on |
| 5 | DimMode[ModeKey] | FactShipment[ModeKey] | * : 1 | Single | Yes | How each leg moved |
| 6 | DimStatus[StatusKey] | FactShipment[StatusKey] | * : 1 | Single | Yes | Whether each leg delivered |

### Part B — the two city roles (answer 1)

**Keep DestCityKey inactive; use USERELATIONSHIP per measure** — consistent with Week 3's
*Weight to Destination*. One physical model, no duplicated dimension: destination analysis goes
through explicit measures, so every destination number is traceable to the inactive relationship
it activates.

In [1]:
# T4 verified against the built .pbix
import pbix_mcp.server as S, json
S.pbix_open("W4_su92-bsdsm-f23-018.pbix")
a = "W4_su92-bsdsm-f23-018"
rels = json.loads(S.pbix_get_model_relationships(a))["message"]
print(rels)
assert rels.count("\n") >= 6 or len([l for l in rels.split("\n") if l.strip()]) == 6
print("6 relationships present; DestCityKey inactive (see register above)")

FromTableName  FromColumnName  ToTableName  ToColumnName  IsActive
FactShipment   DateKey         DimDate      DateKey       1       
FactShipment   OriginCityKey   DimCity      CityKey       1       
FactShipment   DestCityKey     DimCity      CityKey       0       
FactShipment   RouteKey        DimRoute     RouteKey      1       
FactShipment   ModeKey         DimMode      ModeKey       1       
FactShipment   StatusKey       DimStatus    StatusKey     1       
6 relationships present; DestCityKey inactive (see register above)
